# 稀疏估计：压缩感知、OMP、CoSaMP与SAMP算法

**Sparse Estimation: Compressed Sensing, OMP, CoSaMP and SAMP Algorithms**

---

本notebook介绍压缩感知基础理论及其在无线通信中的稀疏信道估计应用。我们将详细讲解OMP（正交匹配追踪）、CoSaMP（压缩采样匹配追踪）和SAMP（稀疏自适应匹配追踪）三大核心算法的原理与实现，并对比它们在不同场景下的性能表现。

## 1. 学习目标 (Objectives)

通过本notebook，您将：

- **理解压缩感知基础**：掌握稀疏性、测量矩阵、RIP条件等核心概念
- **掌握OMP算法**：理解正交匹配追踪的贪婪迭代原理
- **掌握CoSaMP算法**：学习压缩采样匹配追踪的回溯思想
- **掌握SAMP算法**：理解稀疏自适应匹配追踪的阶段切换机制
- **应用于稀疏信道估计**：将压缩感知算法应用于无线信道估计
- **对比分析性能**：通过仿真比较不同算法的估计精度与复杂度

## 2. 压缩感知基础 (Compressed Sensing Fundamentals)

### 2.1 稀疏性 (Sparsity)

稀疏性是压缩感知的核心前提：许多自然信号在某个变换域中只有少数非零系数。

**定义**：如果一个向量 $\mathbf{x} \in \mathbb{C}^N$ 只有 $K \ll N$ 个非零元素，则称 $\mathbf{x}$ 是 $K$-稀疏的，记作 $\\mathbf{x}\|_0 = K$。

**信道的稀疏性**：无线多径信道通常具有稀疏性——能量集中在大约 $L \ll N$ 个路径上，其中 $N$ 是子载波总数。

$$h[n] = \sum_{l=0}^{L-1} \alpha_l \delta[n - \tau_l], \quad L \ll N$$

其中 $\alpha_l$ 是路径增益，$\tau_l$ 是路径延迟。

### 2.2 测量矩阵 (Measurement Matrix)

压缩感知不直接测量稀疏信号，而是通过测量矩阵 $\Phi \in \mathbb{C}^{M \times N}$ 进行降维采样：

$$\mathbf{y} = \mathbf{\Phi} \mathbf{x} + \mathbf{w}$$

其中：
- $\mathbf{y} \in \mathbb{C}^{M}$ 是测量向量（$M < N$）
- $\mathbf{x} \in \mathbb{C}^{N}$ 是稀疏信号
- $\mathbf{w} \in \mathbb{C}^{M}$ 是噪声
- $M$ 远小于 $N$，实现压缩

**关键约束**：为了从 $M$ 个测量值恢复 $N$ 维稀疏信号，必须满足 RIP 条件。

### 2.3 RIP条件 (Restricted Isometry Property)

RIP是衡量测量矩阵恢复稀疏信号能力的核心准则。

**定义**：对于任意 $K$-稀疏向量 $\mathbf{x}$，如果

$$(1 - \delta_K)\|\mathbf{x}\|^2 \leq \|\mathbf{\Phi x}\|^2 \leq (1 + \delta_K)\|\mathbf{x}\|^2$$

对某个 $\delta_K \in (0, 1)$ 成立，则称矩阵 $\mathbf{\Phi}$ 满足 $K$-阶 RIP 常数为 $\delta_K$。

**物理意义**：RIP 要求测量矩阵几乎保持所有稀疏向量的能量，避免对某些稀疏方向过度压缩或放大。

**常用测量矩阵**：
- 随机高斯矩阵 $\Phi_{i,j} \sim \mathcal{N}(0, 1/M)$
- 随机伯努利矩阵 $\Phi_{i,j} = \pm 1/\sqrt{M}$
- 部分傅里叶矩阵（子采样FFT）
- 部分哈达玛矩阵

### 2.4 稀疏恢复问题

稀疏恢复的目标是从测量 $\mathbf{y}$ 恢复稀疏信号 $\mathbf{x}$。

**优化形式**：

\begin{equation*}
\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \|\mathbf{x}\|_0 \quad \text{s.t.} \quad \|\mathbf{y} - \mathbf{\Phi x}\|^2 \leq \epsilon
\end{equation*}

这是 NP-hard 问题（需要遍历所有 $\binom{N}{K}$ 种稀疏组合）。

**实际求解**：
- **贪婪追踪**：OMP、CoSaMP、SAMP —— 迭代选择原子
- **凸优化**：$\ell_1$ 范数替代 $\ell_0$ —— 基追踪去噪 (BPDN)
- **阈值法**：迭代软阈值 (ISTA)、软阈值迭代 (IST)

## 3. OMP（正交匹配追踪）算法 (Orthogonal Matching Pursuit)

### 3.1 OMP算法原理

OMP 是一种贪婪算法，每次迭代选择与当前残差最相关的原子，然后对选中的原子集进行正交投影。

**核心步骤**：

1. **初始化**：残差 $\mathbf{r}^{(0)} = \mathbf{y}$，索引集 $\Lambda^{(0)} = \emptyset$。

2. **匹配**：计算测量矩阵所有列与残差的内积，选择最大值对应的索引：
   $$\lambda_t = \arg\max_{i \notin \Lambda^{(t-1)}} |\langle \mathbf{r}^{(t-1)}, \mathbf{\phi}_i \rangle|$$

3. **更新**：$\Lambda^{(t)} = \Lambda^{(t-1)} \cup \{\lambda_t\}$

4. **正交化**：在选中的原子空间上对信号进行最小二乘投影：
   $$\mathbf{c}^{(t)} = (\mathbf{\Phi}_{\Lambda^{(t)}}^+ \mathbf{y})$$

5. **更新残差**：$\mathbf{r}^{(t)} = \mathbf{y} - \mathbf{\Phi}_{\Lambda^{(t)}} \mathbf{c}^{(t)}$

6. **迭代**：重复步骤 2-5，直到达到稀疏度 $K$ 或残差足够小。

### 3.2 OMP算法流程图

```
输入：测量矩阵 Φ (M×N)，测量向量 y，稀疏度 K
输出：稀疏向量 x_hat (N,)

r ← y                    # 初始化残差
Λ ← ∅                     # 初始化原子索引集
t ← 0

while t < K and ||r|| > ε do
    # 匹配步骤
    for i = 1 to N do
        c_i = |<r, φ_i>|    # 计算内积幅值
    end for
    λ = argmax c_i         # 选择最大相关的原子
    Λ ← Λ ∪ {λ}            # 更新索引集
    
    # 正交投影步骤
    x_Λ = (Φ_Λ^+ y)        # 最小二乘求解选中原子
    
    # 更新残差
    r ← y - Φ_Λ x_Λ
    
    t ← t + 1
end while

x_hat[Λ] = x_Λ            # 在稀疏位置填入估计值
```

### 3.3 OMP算法的收敛性

**正交性保证**：由于每次迭代都对选中原子进行正交投影，残差与所有已选原子正交：

$$\langle \mathbf{r}^{(t)}, \mathbf{\phi}_i \rangle = 0, \quad \forall i \in \Lambda^{(t)}$$

这保证了 OMP 不会重复选择同一个原子，收敛速度更快。

**收敛条件**：当测量矩阵满足 RIP 且 $K < 0.5 / \delta_K$ 时，OMP 能够精确恢复稀疏信号。

## 4. CoSaMP（压缩采样匹配追踪）算法 (Compressive Sampling Matching Pursuit)

### 4.1 CoSaMP算法原理

CoSaMP 是 OMP 的改进版本，每次迭代选择 $2K$ 个相关原子（而非1个），然后通过回溯步骤筛选出最相关的 $K$ 个。这增强了对强相关原子的鲁棒性。

**核心步骤**：

1. **初始化**：残差 $\mathbf{r}^{(0)} = \mathbf{y}$，索引集 $\Lambda^{(0)} = \emptyset$。

2. **匹配**：计算内积，选择幅值最大的 $2K$ 个索引：
   $$J = \arg\max_{S \subset [N], |S|=2K} \sum_{i \in S} |\langle \mathbf{r}^{(t-1)}, \mathbf{\phi}_i \rangle|^2$$

3. **合并**：$\Lambda^{(t)} = \Lambda^{(t-1)} \cup J$

4. **估计**：最小二乘求解：
   $$\mathbf{c}^{(t)} = \arg\min_{\mathbf{x}} \|\mathbf{y} - \mathbf{\Phi}_{\Lambda^{(t)}} \mathbf{x}\|^2$$

5. **回溯**：选择 $\mathbf{c}^{(t)}$ 中幅值最大的 $K$ 个，$\Lambda^{(t)} = \text{supp}(\mathbf{c}^{(t)}_K)$

6. **更新残差**：$\mathbf{r}^{(t)} = \mathbf{y} - \mathbf{\Phi}_{\Lambda^{(t)}} \mathbf{c}^{(t)}$

7. **迭代**：直到收敛。

### 4.2 CoSaMP vs OMP

| 特性 | OMP | CoSaMP |
|------|-----|--------|
| 每次选择的原子数 | 1 | $2K$ |
| 回溯步骤 | 无 | 有 |
| 计算复杂度 | $O(KMN)$ | $O(KMN)$ |
| 收敛速度 | 较快（但每次只增1个） | 较快（每次增2K个，然后筛选） |
| 鲁棒性 | 一般 | 较强 |
| 稀疏度估计 | 需要预先知道 $K$ | 需要预先知道 $K$ |

## 5. SAMP（稀疏自适应匹配追踪）算法 (Sparse Adaptive Matching Pursuit)

### 5.1 SAMP算法原理

SAMP 的核心创新是**阶段切换（Stage Transition）**机制，无需预先知道稀疏度 $K$，能够自适应确定。

**参数**：
- $S$：每阶段选择的原子数（阶段步长）
- 初始阶段 $S = 1$
- 当残差不再显著减小时，增加 $S$

**核心步骤**：

1. **初始化**：残差 $\mathbf{r} = \mathbf{y}$，$S$ = 初始步长，$L$ = 当前估计支撑大小。

2. **匹配**：选择与残差最相关的 $L$ 个原子。

3. **候选集**：$\text{Cand} = \Lambda \cup \text{New}$。

4. **估计与回溯**：
   $$\mathbf{c} = \arg\min_{\mathbf{x}} \|\mathbf{y} - \mathbf{\Phi}_{\text{Cand}} \mathbf{x}\|^2$$
   $$\Lambda = \text{supp}(\mathbf{c}_L)$$  # 选择最大的 $L$ 个

5. **残差计算**：$\mathbf{r}_{new} = \mathbf{y} - \mathbf{\Phi}_{\Lambda} \mathbf{c}$

6. **阶段切换判断**：
   - 如果 $\\mathbf{r}_{new}\_2 \geq \|\mathbf{r}\|_2$：进入下一阶段，$L \leftarrow L + S$
   - 否则：更新残差，继续迭代

7. **迭代**：直到残差足够小或达到最大迭代次数。

### 5.2 SAMP的自适应机制

SAMP 的关键优势是**无需知道稀疏度 $K$**。通过阶段切换机制：

- 当估计的支撑集大小不足时，增加步长 $L$
- 当估计的支撑集大小过大或残差不再下降时，减少 $L$（通过回溯筛选）

这使 SAMP 特别适合实际通信场景，因为实际信道的稀疏度往往是未知或时变的。

## 6. 稀疏信道估计应用 (Sparse Channel Estimation Application)

### 6.1 系统模型

考虑 OFDM 系统中的稀疏信道估计问题。假设信道在时域有 $L$ 个非零点（$L \ll N$）。

**时域信道模型**：
$$\mathbf{h} = [h_0, h_1, ..., h_{N-1}]^T, \quad \|\mathbf{h}\|_0 = L$$

**频域观测**：
$$\mathbf{Y} = \mathbf{X} \cdot \mathbf{F} \mathbf{h} + \mathbf{W} = \mathbf{\Phi} \mathbf{h} + \mathbf{W}$$

其中：
- $\mathbf{F}$ 是 $N \times N$ 的 FFT 矩阵
- $\mathbf{X} = \text{diag}(X_0, ..., X_{N-1})$ 是发送符号对角阵
- $\mathbf{\Phi} = \mathbf{X}\mathbf{F}$ 是等效测量矩阵
- $\mathbf{h}$ 是时域信道冲激响应（稀疏的）

**恢复目标**：从 $\mathbf{Y}$ 和已知 $\mathbf{X}$ 估计稀疏 $\mathbf{h}$。

### 6.2 导频设计与测量矩阵

压缩感知信道估计需要精心设计的导频图案，以满足 RIP 条件。

**常用导频图案**：
- **随机导频**：随机选择子载波放置导频
- **块状导频**：所有子载波在某些符号上发送导频
- **梳状导频**：每隔若干子载波放置导频

**测量矩阵选择**：
当 $\mathbf{X}$ 是随机符号（如 QPSK）时，$\mathbf{\Phi} = \mathbf{X}\mathbf{F}$ 近似满足 RIP，只要导频数量 $M$ 足够多。

**最小导频数**：理论上 $M \geq O(K \log N)$ 即可精确恢复。

## 7. 代码演示：稀疏恢复算法实现 (Implementation)

下面我们实现 OMP、CoSaMP 和 SAMP 算法，并应用于稀疏信道估计。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg
%matplotlib inline

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("稀疏估计演示包加载成功")

### 7.1 系统参数设置

In [ ]:
# OFDM系统参数
N = 64          # 子载波数量（信号维度）
M = 24          # 导频数量（测量数），M < N 实现压缩
L = 4           # 信道实际稀疏度（路径数）
K = 6           # 算法假设的稀疏度（可能不准确）

# 信噪比
SNR_DB = 20     # 信噪比 dB
SNR_LINEAR = 10**(SNR_DB / 10)

print(f"子载波数 N = {N}")
print(f"导频数量 M = {M} (压缩比: {M/N:.2%})")
print(f"信道稀疏度 L = {L}")
print(f"算法假设稀疏度 K = {K}")
print(f"信噪比 SNR = {SNR_DB} dB")

### 7.2 生成稀疏信道和测量矩阵

In [ ]:
def generate_sparse_channel(N, L, seed=42):
    """
    生成稀疏信道
    
    参数:
        N: 信号长度（子载波数）
        L: 稀疏度（路径数）
        seed: 随机种子
    
    返回:
        h: 稀疏时域信道向量 (N,)
    """
    np.random.seed(seed)
    
    # 稀疏信道：只有 L 个非零点
    h = np.zeros(N, dtype=complex)
    
    # 随机选择 L 个位置
    indices = np.random.choice(N, size=L, replace=False)
    indices.sort()
    
    # 生成复高斯路径增益
    amplitudes = (np.random.randn(L) + 1j * np.random.randn(L)) / np.sqrt(2)
    
    for i, idx in enumerate(indices):
        h[idx] = amplitudes[i]
    
    return h, indices


def create_measurement_matrix(N, M, pilot_indices=None, matrix_type='random_gaussian'):
    """
    创建测量矩阵
    
    参数:
        N: 信号维度
        M: 测量数量
        pilot_indices: 导频位置索引（若为None则随机选择）
        matrix_type: 'random_gaussian' 或 'partial_fourier'
    
    返回:
        Phi: 测量矩阵 (M, N)
        pilot_indices: 使用的导频索引
    """
    if pilot_indices is None:
        # 随机选择 M 个位置
        pilot_indices = np.random.choice(N, size=M, replace=False)
    
    if matrix_type == 'random_gaussian':
        # 随机高斯测量矩阵
        Phi = np.random.randn(M, N) + 1j * np.random.randn(M, N)
        Phi = Phi / np.sqrt(M)  # 归一化
        
    elif matrix_type == 'partial_fourier':
        # 部分傅里叶矩阵（子采样FFT）
        FFT = np.fft.fft(np.eye(N)) / np.sqrt(N)
        Phi = FFT[pilot_indices, :]
    
    else:
        raise ValueError(f"未知矩阵类型: {matrix_type}")
    
    return Phi, pilot_indices


# 生成稀疏信道
h_true, true_indices = generate_sparse_channel(N, L, seed=100)
print("稀疏信道生成完成")
print(f"  真实非零位置: {list(true_indices)}")
print(f"  信道能量: {np.linalg.norm(h_true)**2:.4f}")
print(f"  非零元素数量验证: {np.sum(np.abs(h_true) > 1e-6)} = L = {L}")

### 7.3 OMP算法实现

In [ ]:
def omp(Phi, y, K, tol=1e-6, max_iter=None):
    """
    正交匹配追踪（OMP）算法
    
    参数:
        Phi: 测量矩阵 (M, N)
        y: 测量向量 (M,)
        K: 稀疏度
        tol: 残差阈值
        max_iter: 最大迭代次数（默认 K）
    
    返回:
        x_hat: 恢复的稀疏向量 (N,)
        residual_history: 残差范数历史
    """
    M, N = Phi.shape
    if max_iter is None:
        max_iter = K
    
    # 初始化
    residual = y.copy()
    indices = []
    residual_history = [np.linalg.norm(residual)]
    
    # 正交化辅助变量
    Phi_selected = []
    
    for t in range(max_iter):
        # 匹配：计算所有原子与残差的内积幅值
        correlations = np.abs(Phi @ residual)
        
        # 选择最大相关的原子（排除已选）
        correlations[list(indices)] = 0
        idx = np.argmax(correlations)
        
        indices.append(idx)
        Phi_selected.append(Phi[:, idx])
        
        # 正交投影：最小二乘求解
        Phi_s = np.column_stack(Phi_selected)
        x_temp = linalg.lstsq(Phi_s, y)[0]
        
        # 更新残差
        residual = y - Phi_s @ x_temp
        residual_history.append(np.linalg.norm(residual))
        
        # 检查收敛
        if residual_history[-1] < tol:
            break
    
    # 构造稀疏向量
    x_hat = np.zeros(N, dtype=complex)
    if len(indices) > 0:
        x_hat[indices] = x_temp
    
    return x_hat, residual_history, indices


print("OMP算法定义完成")

### 7.4 CoSaMP算法实现

In [ ]:
def cosamp(Phi, y, K, tol=1e-6, max_iter=None):
    """
    压缩采样匹配追踪（CoSaMP）算法
    
    参数:
        Phi: 测量矩阵 (M, N)
        y: 测量向量 (M,)
        K: 稀疏度
        tol: 残差阈值
        max_iter: 最大迭代次数
    
    返回:
        x_hat: 恢复的稀疏向量 (N,)
        residual_history: 残差范数历史
    """
    M, N = Phi.shape
    if max_iter is None:
        max_iter = K * 2
    
    # 初始化
    residual = y.copy()
    indices = []
    residual_history = [np.linalg.norm(residual)]
    
    for t in range(max_iter):
        # 匹配：选择幅值最大的 2K 个原子
        correlations = np.abs(Phi @ residual)
        
        # 标记已选原子（排除）
        mask = np.zeros(N, dtype=bool)
        mask[indices] = True
        correlations[mask] = 0
        
        # 选择 2K 个最大相关的原子
        new_indices = np.argsort(correlations)[-2*K:]
        
        # 合并候选集
        candidate_set = list(set(indices) | set(new_indices))
        
        # 估计：在候选集上做最小二乘
        Phi_cand = Phi[:, candidate_set]
        c = linalg.lstsq(Phi_cand, y)[0]
        
        # 回溯：选择幅值最大的 K 个
        top_k_idx = np.argsort(np.abs(c))[-K:]
        indices = [candidate_set[i] for i in top_k_idx]
        
        # 重新估计
        Phi_s = Phi[:, indices]
        x_temp = linalg.lstsq(Phi_s, y)[0]
        
        # 更新残差
        residual = y - Phi_s @ x_temp
        residual_history.append(np.linalg.norm(residual))
        
        # 检查收敛
        if residual_history[-1] < tol:
            break
    
    # 构造稀疏向量
    x_hat = np.zeros(N, dtype=complex)
    if len(indices) > 0:
        x_hat[indices] = x_temp
    
    return x_hat, residual_history, indices


print("CoSaMP算法定义完成")

### 7.5 SAMP算法实现

In [ ]:
def samp(Phi, y, S=None, tol=1e-6, max_iter=100):
    """
    稀疏自适应匹配追踪（SAMP）算法
    
    参数:
        Phi: 测量矩阵 (M, N)
        y: 测量向量 (M,)
        S: 阶段步长（默认 sqrt(N)）
        tol: 残差阈值
        max_iter: 最大迭代次数
    
    返回:
        x_hat: 恢复的稀疏向量 (N,)
        residual_history: 残差范数历史
        final_L: 最终支撑大小
    """
    M, N = Phi.shape
    if S is None:
        S = max(1, int(np.sqrt(N)))
    
    # 初始化
    residual = y.copy()
    L = S  # 当前估计的支撑大小
    stage = 1
    indices = []
    residual_history = [np.linalg.norm(residual)]
    
    for it in range(max_iter):
        # 匹配：选择幅值最大的 L 个原子
        correlations = np.abs(Phi @ residual)
        
        # 选择 L 个最大相关的原子（排除已选）
        temp_indices = np.argsort(correlations)[-L:]
        candidate_set = list(set(indices) | set(temp_indices))
        
        # 估计：在候选集上做最小二乘
        Phi_cand = Phi[:, candidate_set]
        c = linalg.lstsq(Phi_cand, y)[0]
        
        # 回溯：选择幅值最大的 L 个
        top_L_idx = np.argsort(np.abs(c))[-L:]
        new_indices = [candidate_set[i] for i in top_L_idx]
        
        # 重新估计
        Phi_new = Phi[:, new_indices]
        x_new = linalg.lstsq(Phi_new, y)[0]
        
        # 计算新残差
        residual_new = y - Phi_new @ x_new
        residual_norm_new = np.linalg.norm(residual_new)
        residual_norm_old = residual_history[-1]
        
        # 阶段切换判断
        if residual_norm_new >= residual_norm_old:
            # 残差没有减少，进入下一阶段
            stage += 1
            L = min(L + S, N)
        else:
            # 残差减少，更新支撑集
            indices = new_indices
            residual = residual_new
            residual_history.append(residual_norm_new)
            
            # 检查收敛
            if residual_norm_new < tol:
                break
    
    # 构造稀疏向量
    x_hat = np.zeros(N, dtype=complex)
    if len(indices) > 0:
        x_hat[indices] = x_new
    
    return x_hat, residual_history, indices, L


print("SAMP算法定义完成")

### 7.6 稀疏恢复演示

In [ ]:
# 创建测量矩阵（随机高斯）
np.random.seed(50)
pilot_indices = np.random.choice(N, size=M, replace=False)
Phi, _ = create_measurement_matrix(N, M, pilot_indices, matrix_type='random_gaussian')

# 计算测量值 y = Phi * h + noise
y = Phi @ h_true

# 添加噪声
signal_power = np.mean(np.abs(y)**2)
noise_power = signal_power / SNR_LINEAR
noise = np.sqrt(noise_power / 2) * (np.random.randn(M) + 1j * np.random.randn(M))
y_noisy = y + noise

# 使用不同算法恢复
h_omp, res_omp, idx_omp = omp(Phi, y_noisy, K=L, tol=1e-8)
h_cosamp, res_cosamp, idx_cosamp = cosamp(Phi, y_noisy, K=L, tol=1e-8)
h_samp, res_samp, idx_samp, final_L = samp(Phi, y_noisy, S=2, tol=1e-8)

# 计算恢复误差
mse_omp = np.linalg.norm(h_omp - h_true)**2 / np.linalg.norm(h_true)**2
mse_cosamp = np.linalg.norm(h_cosamp - h_true)**2 / np.linalg.norm(h_true)**2
mse_samp = np.linalg.norm(h_samp - h_true)**2 / np.linalg.norm(h_true)**2

print(f"稀疏恢复结果 (SNR={SNR_DB}dB, L={L}, M={M}):")
print(f"  OMP:    MSE = {mse_omp:.6f}, 估计支撑 = {idx_omp}")
print(f"  CoSaMP: MSE = {mse_cosamp:.6f}, 估计支撑 = {idx_cosamp}")
print(f"  SAMP:   MSE = {mse_samp:.6f}, 估计支撑 = {idx_samp}, 最终L={final_L}")
print(f"  真实支撑 = {list(true_indices)}")

### 7.7 恢复结果可视化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 子图1：真实信道
ax1 = axes[0, 0]
ax1.stem(range(N), np.abs(h_true), linefmt='C0-', markerfmt='C0o', basefmt='k-')
ax1.set_xlabel('索引', fontsize=11)
ax1.set_ylabel('幅度', fontsize=11)
ax1.set_title(f'真实稀疏信道 (L={L})', fontsize=12)
ax1.grid(True, alpha=0.3)

# 子图2：OMP恢复
ax2 = axes[0, 1]
ax2.stem(range(N), np.abs(h_omp), linefmt='C1-', markerfmt='C1o', basefmt='k-')
ax2.set_xlabel('索引', fontsize=11)
ax2.set_ylabel('幅度', fontsize=11)
ax2.set_title(f'OMP恢复 (MSE={mse_omp:.4f})', fontsize=12)
ax2.grid(True, alpha=0.3)

# 子图3：CoSaMP恢复
ax3 = axes[1, 0]
ax3.stem(range(N), np.abs(h_cosamp), linefmt='C2-', markerfmt='C2o', basefmt='k-')
ax3.set_xlabel('索引', fontsize=11)
ax3.set_ylabel('幅度', fontsize=11)
ax3.set_title(f'CoSaMP恢复 (MSE={mse_cosamp:.4f})', fontsize=12)
ax3.grid(True, alpha=0.3)

# 子图4：SAMP恢复
ax4 = axes[1, 1]
ax4.stem(range(N), np.abs(h_samp), linefmt='C3-', markerfmt='C3o', basefmt='k-')
ax4.set_xlabel('索引', fontsize=11)
ax4.set_ylabel('幅度', fontsize=11)
ax4.set_title(f'SAMP恢复 (MSE={mse_samp:.4f})', fontsize=12)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察：")
print("1. 所有算法都能准确恢复稀疏信道的非零位置")
print("2. 恢复幅度与真实值有一定误差（噪声影响）")
print("3. SAMP不需要预先知道稀疏度L，能自适应确定")

## 8. 残差收敛分析 (Convergence Analysis)

下面比较不同算法的残差收敛速度。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.semilogy(range(len(res_omp)), res_omp, 'b-o', linewidth=2, markersize=6, label='OMP')
ax.semilogy(range(len(res_cosamp)), res_cosamp, 'r-s', linewidth=2, markersize=6, label='CoSaMP')
ax.semilogy(range(len(res_samp)), res_samp, 'g-^', linewidth=2, markersize=6, label='SAMP')

ax.set_xlabel('迭代次数', fontsize=12)
ax.set_ylabel('残差范数 (log scale)', fontsize=12)
ax.set_title(f'残差收敛曲线 (SNR={SNR_DB}dB)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察：")
print(f"  OMP:     {len(res_omp)} 次迭代收敛")
print(f"  CoSaMP:  {len(res_cosamp)} 次迭代收敛")
print(f"  SAMP:    {len(res_samp)} 次迭代收敛")

## 9. 不同SNR下的性能对比 (Performance vs SNR)

下面比较三种算法在不同信噪比下的恢复性能。

In [ ]:
def test_sparse_recovery(N, M, L, snr_db, num_trials=50):
    """
    测试稀疏恢复算法在不同SNR下的性能
    
    返回:
        mse_omp, mse_cosamp, mse_samp: 各算法的MSE
    """
    mse_omp_all = []
    mse_cosamp_all = []
    mse_samp_all = []
    
    for trial in range(num_trials):
        # 生成信道
        np.random.seed(trial * 100)
        h, true_idx = generate_sparse_channel(N, L, seed=trial)
        
        # 创建测量矩阵
        np.random.seed(trial * 200)
        pilot_indices = np.random.choice(N, size=M, replace=False)
        Phi, _ = create_measurement_matrix(N, M, pilot_indices)
        
        # 测量
        y = Phi @ h
        
        # 加噪
        signal_power = np.mean(np.abs(y)**2)
        snr_lin = 10**(snr_db / 10)
        noise_power = signal_power / snr_lin
        noise = np.sqrt(noise_power / 2) * (np.random.randn(M) + 1j * np.random.randn(M))
        y_noisy = y + noise
        
        # OMP
        h_omp, _, _ = omp(Phi, y_noisy, K=L)
        mse_omp_all.append(np.linalg.norm(h_omp - h)**2 / np.linalg.norm(h)**2)
        
        # CoSaMP
        h_cosamp, _, _ = cosamp(Phi, y_noisy, K=L)
        mse_cosamp_all.append(np.linalg.norm(h_cosamp - h)**2 / np.linalg.norm(h)**2)
        
        # SAMP
        h_samp, _, _, _ = samp(Phi, y_noisy, S=2)
        mse_samp_all.append(np.linalg.norm(h_samp - h)**2 / np.linalg.norm(h)**2)
    
    return np.mean(mse_omp_all), np.mean(mse_cosamp_all), np.mean(mse_samp_all)


# 测试不同SNR
snr_range = [5, 10, 15, 20, 25, 30]
num_trials = 50

print("开始SNR性能仿真...")
results = []
for snr_db in snr_range:
    mse_omp, mse_cosamp, mse_samp = test_sparse_recovery(N, M, L, snr_db, num_trials)
    results.append((snr_db, mse_omp, mse_cosamp, mse_samp))
    print(f"SNR = {snr_db:2d} dB | OMP: {mse_omp:.6f} | CoSaMP: {mse_cosamp:.6f} | SAMP: {mse_samp:.6f}")

print("\n仿真完成!")

In [ ]:
# 绘制性能曲线
snrs = [r[0] for r in results]
mse_omp_all = [r[1] for r in results]
mse_cosamp_all = [r[2] for r in results]
mse_samp_all = [r[3] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：线性坐标
ax1 = axes[0]
ax1.plot(snrs, mse_omp_all, 'b-o', linewidth=2, markersize=8, label='OMP')
ax1.plot(snrs, mse_cosamp_all, 'r-s', linewidth=2, markersize=8, label='CoSaMP')
ax1.plot(snrs, mse_samp_all, 'g-^', linewidth=2, markersize=8, label='SAMP')
ax1.set_xlabel('信噪比 (dB)', fontsize=12)
ax1.set_ylabel('归一化MSE', fontsize=12)
ax1.set_title('稀疏恢复性能 vs SNR（线性坐标）', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# 右图：对数坐标
ax2 = axes[1]
ax2.semilogy(snrs, mse_omp_all, 'b-o', linewidth=2, markersize=8, label='OMP')
ax2.semilogy(snrs, mse_cosamp_all, 'r-s', linewidth=2, markersize=8, label='CoSaMP')
ax2.semilogy(snrs, mse_samp_all, 'g-^', linewidth=2, markersize=8, label='SAMP')
ax2.set_xlabel('信噪比 (dB)', fontsize=12)
ax2.set_ylabel('归一化MSE (log)', fontsize=12)
ax2.set_title('稀疏恢复性能 vs SNR（对数坐标）', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察：")
print("1. 所有算法的MSE随SNR提高而降低")
print("2. CoSaMP在高噪声环境下表现通常更稳定")
print("3. SAMP无需预先知道稀疏度，在稀疏度未知场景下有优势")

## 10. 稀疏度估计准确性分析 (Sparsity Estimation Accuracy)

SAMP的优势在于不需要预先知道稀疏度，下面验证其自适应能力。

In [ ]:
def test_sparsity_adaptation(N, M, L_true, snr_db, num_trials=30):
    """
    测试算法在错误假设稀疏度下的表现
    
    L假设的稀疏度范围
    """
    L_range = [2, 3, 4, 5, 6, 8, 10]
    
    results_omp = {L_guess: [] for L_guess in L_range}
    results_cosamp = {L_guess: [] for L_guess in L_range}
    
    for trial in range(num_trials):
        # 生成信道
        h, _ = generate_sparse_channel(N, L_true, seed=trial)
        
        # 创建测量矩阵
        pilot_indices = np.random.choice(N, size=M, replace=False)
        Phi, _ = create_measurement_matrix(N, M, pilot_indices)
        
        # 测量并加噪
        y = Phi @ h
        signal_power = np.mean(np.abs(y)**2)
        noise_power = signal_power / 10**(snr_db/10)
        noise = np.sqrt(noise_power/2) * (np.random.randn(M) + 1j*np.random.randn(M))
        y_noisy = y + noise
        
        # 测试不同假设稀疏度
        for L_guess in L_range:
            # OMP
            h_omp, _, _ = omp(Phi, y_noisy, K=L_guess)
            mse = np.linalg.norm(h_omp - h)**2 / np.linalg.norm(h)**2
            results_omp[L_guess].append(mse)
            
            # CoSaMP
            h_cosamp, _, _ = cosamp(Phi, y_noisy, K=L_guess)
            mse = np.linalg.norm(h_cosamp - h)**2 / np.linalg.norm(h)**2
            results_cosamp[L_guess].append(mse)
    
    # 平均
    mean_omp = {L_guess: np.mean(vals) for L_guess, vals in results_omp.items()}
    mean_cosamp = {L_guess: np.mean(vals) for L_guess, vals in results_cosamp.items()}
    
    return L_range, mean_omp, mean_cosamp


# 测试稀疏度敏感性
L_true = 4
snr_test = 20

print(f"测试稀疏度敏感性（真实L={L_true}, SNR={snr_test}dB）...")
L_range, mean_omp, mean_cosamp = test_sparsity_adaptation(N, M, L_true, snr_test, num_trials=30)

# SAMP自适应结果
samp_mse = []
for trial in range(30):
    h, _ = generate_sparse_channel(N, L_true, seed=trial)
    pilot_indices = np.random.choice(N, size=M, replace=False)
    Phi, _ = create_measurement_matrix(N, M, pilot_indices)
    y = Phi @ h
    signal_power = np.mean(np.abs(y)**2)
    noise_power = signal_power / 10**(snr_test/10)
    noise = np.sqrt(noise_power/2) * (np.random.randn(M) + 1j*np.random.randn(M))
    y_noisy = y + noise
    h_samp, _, _, _ = samp(Phi, y_noisy, S=2)
    samp_mse.append(np.linalg.norm(h_samp - h)**2 / np.linalg.norm(h)**2)

print("SAMP平均MSE（自适应）:", np.mean(samp_mse))

# 绘制
fig, ax = plt.subplots(figsize=(10, 6))

ax.semilogy(L_range, [mean_omp[L] for L in L_range], 'b-o', linewidth=2, markersize=8, label='OMP')
ax.semilogy(L_range, [mean_cosamp[L] for L in L_range], 'r-s', linewidth=2, markersize=8, label='CoSaMP')
ax.axhline(y=np.mean(samp_mse), color='g', linestyle='--', linewidth=2, label=f'SAMP (自适应, MSE={np.mean(samp_mse):.4f})')
ax.axvline(x=L_true, color='k', linestyle=':', linewidth=2, label=f'真实稀疏度 L={L_true}')

ax.set_xlabel('假设的稀疏度 K', fontsize=12)
ax.set_ylabel('归一化MSE', fontsize=12)
ax.set_title(f'稀疏度估计准确性分析 (真实L={L_true}, SNR={snr_test}dB)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察：")
print("1. 当假设的稀疏度接近真实值时，OMP和CoSaMP性能最好")
print("2. 假设稀疏度过小会导致欠估计，过大可能导致过拟合")
print("3. SAMP能够自适应找到正确的稀疏度，性能稳定")

## 11. 测量数对恢复性能的影响 (Measurement Number vs Performance)

压缩感知要求测量数 M 满足一定条件才能恢复稀疏信号。下面分析 M 的影响。

In [ ]:
def test_measurement_effect(N, L, M_range, snr_db, num_trials=30):
    """
    测试不同测量数M对恢复性能的影响
    """
    results = {'omp': [], 'cosamp': [], 'samp': []}
    
    for M in M_range:
        mse_omp = []
        mse_cosamp = []
        mse_samp = []
        
        for trial in range(num_trials):
            # 生成信道
            h, _ = generate_sparse_channel(N, L, seed=trial)
            
            # 测量矩阵
            pilot_indices = np.random.choice(N, size=M, replace=False)
            Phi, _ = create_measurement_matrix(N, M, pilot_indices)
            
            # 测量
            y = Phi @ h
            signal_power = np.mean(np.abs(y)**2)
            noise_power = signal_power / 10**(snr_db/10)
            noise = np.sqrt(noise_power/2) * (np.random.randn(M) + 1j*np.random.randn(M))
            y_noisy = y + noise
            
            # OMP
            h_omp, _, _ = omp(Phi, y_noisy, K=L)
            mse_omp.append(np.linalg.norm(h_omp - h)**2 / np.linalg.norm(h)**2)
            
            # CoSaMP
            h_cosamp, _, _ = cosamp(Phi, y_noisy, K=L)
            mse_cosamp.append(np.linalg.norm(h_cosamp - h)**2 / np.linalg.norm(h)**2)
            
            # SAMP
            h_samp, _, _, _ = samp(Phi, y_noisy, S=2)
            mse_samp.append(np.linalg.norm(h_samp - h)**2 / np.linalg.norm(h)**2)
        
        results['omp'].append(np.mean(mse_omp))
        results['cosamp'].append(np.mean(mse_cosamp))
        results['samp'].append(np.mean(mse_samp))
        print(f"M={M:2d} | OMP: {results['omp'][-1]:.6f} | CoSaMP: {results['cosamp'][-1]:.6f} | SAMP: {results['samp'][-1]:.6f}")
    
    return results


# 测试M的影响
M_range = [8, 12, 16, 20, 24, 28, 32]
snr_test = 20

print(f"测量数M对恢复性能的影响 (SNR={snr_test}dB, L={L}):")
results_M = test_measurement_effect(N, L, M_range, snr_test, num_trials=30)

# 绘制
fig, ax = plt.subplots(figsize=(10, 6))

ax.semilogy(M_range, results_M['omp'], 'b-o', linewidth=2, markersize=8, label='OMP')
ax.semilogy(M_range, results_M['cosamp'], 'r-s', linewidth=2, markersize=8, label='CoSaMP')
ax.semilogy(M_range, results_M['samp'], 'g-^', linewidth=2, markersize=8, label='SAMP')

# 理论下限：M > c * K * log(N)
c = 2  # 常数因子
M_min = c * L * np.log(N)
ax.axvline(x=M_min, color='k', linestyle='--', linewidth=2, label=f'理论下限 M≈{M_min:.1f}')

ax.set_xlabel('测量数 M', fontsize=12)
ax.set_ylabel('归一化MSE', fontsize=12)
ax.set_title(f'测量数对稀疏恢复性能的影响 (SNR={snr_test}dB, L={L})', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察：")
print(f"1. 恢复性能随M增加而提升（更多测量带来更多信息）")
print(f"2. 理论下限M≈{M_min:.1f}，当M接近此值时性能开始显著改善")
print("3. M过大时，增加测量数的边际收益减小")

## 12. 算法复杂度对比 (Complexity Comparison)

下面比较三种算法的计算复杂度。

In [ ]:
def benchmark_algorithms(N, M, L, num_iter=100):
    """
    基准测试算法运行时间
    """
    import time
    
    # 创建测试数据
    np.random.seed(42)
    Phi = np.random.randn(M, N) + 1j * np.random.randn(M, N)
    h = np.zeros(N, dtype=complex)
    h[np.random.choice(N, size=L, replace=False)] = np.random.randn(L) + 1j * np.random.randn(L)
    y = Phi @ h + 0.01 * (np.random.randn(M) + 1j * np.random.randn(M))
    
    # 测试OMP
    start = time.time()
    for _ in range(num_iter):
        omp(Phi, y, K=L)
    time_omp = (time.time() - start) / num_iter
    
    # 测试CoSaMP
    start = time.time()
    for _ in range(num_iter):
        cosamp(Phi, y, K=L)
    time_cosamp = (time.time() - start) / num_iter
    
    # 测试SAMP
    start = time.time()
    for _ in range(num_iter):
        samp(Phi, y, S=2)
    time_samp = (time.time() - start) / num_iter
    
    return time_omp, time_cosamp, time_samp


# 基准测试
print("算法运行时间基准测试...")
time_omp, time_cosamp, time_samp = benchmark_algorithms(N, M, L, num_iter=200)

print(f"N={N}, M={M}, L={L}:")
print(f"  OMP:    {time_omp*1000:.3f} ms/次")
print(f"  CoSaMP: {time_cosamp*1000:.3f} ms/次")
print(f"  SAMP:   {time_samp*1000:.3f} ms/次")

# 复杂度分析图
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['OMP', 'CoSaMP', 'SAMP']
times = [time_omp * 1000, time_cosamp * 1000, time_samp * 1000]
colors = ['steelblue', 'coral', 'seagreen']

bars = ax.bar(methods, times, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('运行时间 (ms)', fontsize=12)
ax.set_title(f'算法运行时间对比 (N={N}, M={M}, L={L})', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

# 添加数值标签
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{t:.2f}', 
            ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

print("观察：")
print("1. OMP复杂度最低，每次只选择一个原子")
print("2. CoSaMP需要更复杂的回溯操作，耗时略高")
print("3. SAMP需要动态调整支撑大小，迭代次数可能更多")

## 13. 实际信道估计应用 (Practical Channel Estimation)

下面展示如何将压缩感知算法应用于实际的OFDM信道估计。

In [ ]:
def cs_channel_estimation_ofdm(N, M_pilot, L, snr_db, method='omp'):
    """
    基于压缩感知的OFDM信道估计
    
    参数:
        N: 子载波数
        M_pilot: 导频数
        L: 信道稀疏度
        snr_db: 信噪比
        method: 'omp', 'cosamp', 或 'samp'
    
    返回:
        mse: 均方误差
        h_est: 估计的信道
        h_true: 真实信道
    """
    # 生成稀疏信道（时域）
    h_time, idx_true = generate_sparse_channel(N, L)
    
    # FFT得到频域信道
    H_freq = np.fft.fft(h_time)
    
    # 随机导频位置
    np.random.seed(200)
    pilot_indices = np.random.choice(N, size=M_pilot, replace=False)
    
    # 导频符号（QPSK）
    qpsk = np.array([1+0j, 0+1j, -1+0j, 0-1j]) / np.sqrt(2)
    X_pilot = np.random.choice(qpsk, size=M_pilot)
    
    # 频域观测 Y = X * H + W
    Y_pilot = X_pilot * H_freq[pilot_indices]
    
    # 加噪声
    signal_power = np.mean(np.abs(Y_pilot)**2)
    noise_power = signal_power / 10**(snr_db/10)
    noise = np.sqrt(noise_power/2) * (np.random.randn(M_pilot) + 1j * np.random.randn(M_pilot))
    Y_pilot_noisy = Y_pilot + noise
    
    # 构建等效测量矩阵 Phi = X * FFT（只在导频位置）
    # 实际上 Y = diag(X) * FFT_{pilot_indices} * h_time
    FFT_pilot = np.fft.fft(np.eye(N))[pilot_indices, :]
    Phi = np.diag(X_pilot) @ FFT_pilot
    
    # 稀疏恢复：估计时域信道 h_time
    if method == 'omp':
        h_est, _, _ = omp(Phi, Y_pilot_noisy, K=L)
    elif method == 'cosamp':
        h_est, _, _ = cosamp(Phi, Y_pilot_noisy, K=L)
    elif method == 'samp':
        h_est, _, _, _ = samp(Phi, Y_pilot_noisy, S=2)
    else:
        raise ValueError(f"未知方法: {method}")
    
    # 恢复频域信道
    H_est_freq = np.fft.fft(h_est)
    
    # 计算MSE
    mse = np.mean(np.abs(H_est_freq - H_freq)**2) / np.mean(np.abs(H_freq)**2)
    
    return mse, H_est_freq, H_freq


# 测试OFDM信道估计
snr_test = 20
M_pilot = 20

print(f"OFDM稀疏信道估计演示 (SNR={snr_test}dB, M_pilot={M_pilot}, L={L}):")
print()

mse_omp, H_est_omp, H_true = cs_channel_estimation_ofdm(N, M_pilot, L, snr_test, 'omp')
print(f"  OMP:     MSE = {mse_omp:.6f}")

mse_cosamp, H_est_cosamp, _ = cs_channel_estimation_ofdm(N, M_pilot, L, snr_test, 'cosamp')
print(f"  CoSaMP:  MSE = {mse_cosamp:.6f}")

mse_samp, H_est_samp, _ = cs_channel_estimation_ofdm(N, M_pilot, L, snr_test, 'samp')
print(f"  SAMP:    MSE = {mse_samp:.6f}")

### 13.1 频域信道估计结果对比

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

subcarriers = np.arange(N)

# 真实频域信道
ax1 = axes[0, 0]
ax1.plot(subcarriers, np.abs(H_true), 'b-', linewidth=2)
ax1.set_xlabel('子载波索引', fontsize=11)
ax1.set_ylabel('幅度', fontsize=11)
ax1.set_title('真实频域信道响应', fontsize=12)
ax1.grid(True, alpha=0.3)

# OMP恢复
ax2 = axes[0, 1]
ax2.plot(subcarriers, np.abs(H_est_omp), 'r-', linewidth=2, label='OMP恢复')
ax2.plot(subcarriers, np.abs(H_true), 'b--', linewidth=1.5, label='真实值')
ax2.set_xlabel('子载波索引', fontsize=11)
ax2.set_ylabel('幅度', fontsize=11)
ax2.set_title(f'OMP信道估计 (MSE={mse_omp:.4f})', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# CoSaMP恢复
ax3 = axes[1, 0]
ax3.plot(subcarriers, np.abs(H_est_cosamp), 'g-', linewidth=2, label='CoSaMP恢复')
ax3.plot(subcarriers, np.abs(H_true), 'b--', linewidth=1.5, label='真实值')
ax3.set_xlabel('子载波索引', fontsize=11)
ax3.set_ylabel('幅度', fontsize=11)
ax3.set_title(f'CoSaMP信道估计 (MSE={mse_cosamp:.4f})', fontsize=12)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# SAMP恢复
ax4 = axes[1, 1]
ax4.plot(subcarriers, np.abs(H_est_samp), 'm-', linewidth=2, label='SAMP恢复')
ax4.plot(subcarriers, np.abs(H_true), 'b--', linewidth=1.5, label='真实值')
ax4.set_xlabel('子载波索引', fontsize=11)
ax4.set_ylabel('幅度', fontsize=11)
ax4.set_title(f'SAMP信道估计 (MSE={mse_samp:.4f})', fontsize=12)
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察：")
print("1. 所有压缩感知算法都能有效估计稀疏信道")
print("2. 估计的频域响应与真实值高度吻合")
print("3. 利用信道稀疏性，即使导频数量M远小于子载波数N也能准确恢复")

## 14. 方法比较总结 (Method Comparison Summary)

### 14.1 算法特性对比

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')

# 表格数据
table_data = [
    ['每次选择原子数', '1', '2K', '自适应 (S)'],
    ['是否需要知道K', '是', '是', '否'],
    ['回溯步骤', '无', '有', '有'],
    ['计算复杂度', 'O(KMN)', 'O(KMN)', 'O(KMN) ~ O(K²N)'],
    ['收敛速度', '较慢（每次1个）', '较快（每次2K个）', '中等'],
    ['鲁棒性', '一般', '较强', '强'],
    ['适用场景', '稀疏度已知', '稀疏度已知', '稀疏度未知或时变']
]

table = ax.table(
    cellText=table_data,
    rowLabels=['每次选择原子数', '是否需要知道K', '回溯步骤', '计算复杂度', '收敛速度', '鲁棒性', '适用场景'],
    colLabels=['', 'OMP', 'CoSaMP', 'SAMP'],
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

# 表头样式
for i in range(4):
    table[(0, i)].set_facecolor('#4472C4')
    table[(0, i)].set_text_props(color='white', fontweight='bold')

ax.set_title('OMP、CoSaMP、SAMP 算法特性对比', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

### 14.2 应用场景建议

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')

scenarios = [
    ['场景', '推荐算法', '原因'],
    ['稀疏度K已知且固定', 'OMP', '简单高效，复杂度最低'],
    ['稀疏度K已知但有噪声', 'CoSaMP', '回溯机制增强鲁棒性'],
    ['稀疏度K未知或时变', 'SAMP', '自适应确定稀疏度'],
    ['高速移动信道（时变稀疏度）', 'SAMP', '自适应跟踪稀疏度变化'],
    ['计算资源受限', 'OMP', '单原子选择，运算量最小'],
    ['高信噪比环境', 'OMP 或 CoSaMP', '差异不大，根据K是否已知选择'],
]

table = ax.table(
    cellText=scenarios[1:],
    colLabels=scenarios[0],
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

# 表头样式
for i in range(3):
    table[(0, i)].set_facecolor('#4472C4')
    table[(0, i)].set_text_props(color='white', fontweight='bold')

ax.set_title('稀疏恢复算法应用场景建议', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

## 15. 思考题 (Review Questions)

1. **压缩感知前提**：解释为什么稀疏性是压缩感知的核心前提。如果信号不稀疏，压缩感知方法还能有效恢复信号吗？有什么替代方案？

2. **RIP条件的意义**：RIP条件确保测量矩阵不会过度压缩稀疏信号。请分析：
   - RIP常数 $\delta_K$ 的大小对恢复性能有何影响？
   - 为什么随机高斯矩阵以高概率满足RIP？
   - 如果测量矩阵不满足RIP，会发生什么问题？

3. **OMP vs CoSaMP**：比较OMP和CoSaMP的优缺点。CoSaMP每次选择$2K$个原子再回溯到$K$个的设计有什么优势？在什么情况下CoSaMP可能优于OMP？

4. **SAMP的自适应机制**：SAMP通过阶段切换机制自适应确定稀疏度。请分析：
   - 当残差不再减小时，为什么要增加$L$而不是继续迭代？
   - 如果步长$S$设置得过大或过小，会有什么问题？
   - 如何选择合适的$S$值？

5. **测量数与稀疏度**：理论上，压缩感知需要$M \geq O(K \log N)$个测量值才能精确恢复。请分析：
   - 如果$M$小于这个下界会发生什么？
   - 实际中如何确定最小的安全$M$值？
   - 增加$M$到远超过下界会有什么收益和代价？

6. **噪声敏感性**：在低SNR条件下，稀疏恢复算法的性能会下降。请分析：
   - 噪声如何影响贪婪追踪算法的选择过程？
   - 如果测量矩阵的列之间存在强相关性（相干性高），会有什么影响？
   - 如何提高算法在低SNR下的鲁棒性？

7. **信道估计应用**：将压缩感知应用于OFDM信道估计时：
   - 为什么时域信道冲激响应是稀疏的（稀疏度$L \ll N$）？
   - 导频图案设计如何影响测量矩阵的RIP性质？
   - 与传统LS估计相比，压缩感知方法的主要优势是什么？
   - 在什么场景下压缩感知信道估计优势最明显？

8. **算法复杂度**：假设$N=1024$，$M=256$，$K=8$，请估算OMP、CoSaMP和SAMP各需要多少次矩阵-向量乘法，以及存储需求。

9. **实际挑战**：在实际无线通信系统中部署压缩感知信道估计时，可能面临哪些实际问题？例如：导频开销、信道时变性、估计延迟等。如何解决这些问题？

10. **深度扩展**：如果你要实现一个基于压缩感知的5G/6G信道估计方案，请设计：
    - 导频图案（位置和数量）
    - 测量矩阵类型
    - 算法选择和参数配置
    - 与现有5G/6G标准的兼容性

---

## 总结 (Summary)

本notebook系统介绍了压缩感知基础理论及其在稀疏信道估计中的应用：

- **压缩感知基础**：稀疏性是核心前提，RIP条件保证恢复可行性
- **OMP算法**：单原子贪婪选择，每次迭代正交投影，简洁高效
- **CoSaMP算法**：$2K$选择+$K$回溯，增强鲁棒性，收敛更快
- **SAMP算法**：阶段切换机制，无需预先知道稀疏度，自适应能力强
- **稀疏信道估计**：利用信道时域稀疏性，减少导频开销，提升估计精度

关键洞察：**在稀疏场景下，压缩感知能够以远低于奈奎斯特采样率的测量数恢复信号，这为5G/6G高载波频率、大带宽系统中的信道估计提供了理论支撑和实用方案。**